[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/10_Deployment/01_Edge_Deployment/Edge_Deployment_Deep_Dive.ipynb)

# Edge Deployment with ONNX Runtime — Deep Dive

A comprehensive treatment of deploying ONNX models on resource-constrained edge devices:
Raspberry Pi, NVIDIA Jetson, Intel Neural Compute Stick, and custom embedded platforms.

---

## Table of Contents

| # | Section | Key Topics |
|---|---------|------------|
| 1 | [Edge Computing Fundamentals](#1) | Definition, latency analysis, power constraints |
| 2 | [Memory Budget Mathematics](#2) | Tensor memory, activation maps, peak allocation |
| 3 | [Power and Thermal Modeling](#3) | CMOS power equation, thermal throttling |
| 4 | [Compute Roofline Analysis](#4) | Arithmetic intensity, memory bandwidth |
| 5 | [Quantization for Edge](#5) | INT8 math, calibration theory, error analysis |
| 6 | [Hardware Platform Deep Dive](#6) | Raspberry Pi, Jetson, Intel VPU |
| 7 | [ORT Execution Providers](#7) | Provider selection, fallback chains |
| 8 | [Latency Decomposition](#8) | Pre/inference/post breakdown, pipelining |
| 9 | [Deployment Architecture Patterns](#9) | Streaming, batched, multi-model |
| 10 | [Reliability and Monitoring](#10) | Watchdogs, drift detection, OTA updates |

---

<a id='1'></a>
## 1. Edge Computing Fundamentals

Edge computing places inference **at or near the data source** — cameras, sensors, industrial controllers — rather than requiring a round-trip to a cloud data center. This architectural decision fundamentally reshapes the constraints under which a model must operate.

The latency budget for edge inference is typically measured in single-digit to tens of milliseconds. Consider an autonomous robot that must detect obstacles: if the camera runs at 30 FPS, each frame arrives every 33.3 ms. The entire pipeline — capture, preprocess, infer, postprocess, actuate — must fit within this budget:

$$T_{\text{total}} = T_{\text{capture}} + T_{\text{preprocess}} + T_{\text{inference}} + T_{\text{postprocess}} + T_{\text{actuation}} \leq \frac{1}{\text{FPS}}$$

Unlike cloud deployments where you can throw more hardware at latency problems, edge devices have **fixed resource envelopes**. You cannot add more RAM to a Raspberry Pi in production. You cannot attach a bigger GPU to a Jetson Nano. The model must fit the hardware — not the other way around.

### The Edge Deployment Triangle

```
                    Accuracy
                      /\
                     /  \
                    /    \
                   / PICK \
                  /  TWO   \
                 /          \
                /____________\
         Latency              Power
```

Every edge deployment navigates this triangle: you can optimize for any two axes, but the third suffers. A highly accurate model may be too slow or too power-hungry. A fast model within power budget may sacrifice accuracy. Understanding the mathematical relationships between these axes is critical for making informed trade-offs.

### Key Differentiators from Cloud

| Dimension | Cloud | Edge |
|-----------|-------|------|
| **Batch size** | 32–256 | 1 (streaming) |
| **Memory** | 64–512 GB | 1–8 GB |
| **Compute** | A100/H100 (312 TFLOPS) | 1–20 TOPS |
| **Power** | 300–700W per GPU | 5–30W total system |
| **Connectivity** | Always-on datacenter network | Intermittent, metered |
| **Updates** | Blue-green deploy in seconds | OTA with rollback strategy |

<a id='2'></a>
## 2. Memory Budget Mathematics

On edge devices, memory is the **primary constraint**. A model must fit its parameters, activations, and runtime overhead within the device's physical memory. Let us formalize this.

### Total Memory Footprint

The total memory required by a model during inference is:

$$M_{\text{total}} = M_{\text{params}} + M_{\text{activations}} + M_{\text{runtime}}$$

where:

$$M_{\text{params}} = \sum_{i=1}^{L} \text{numel}(W_i) \cdot \text{sizeof}(\text{dtype}_i)$$

For a convolutional layer with kernel $W \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times k_h \times k_w}$:

$$M_{\text{conv}} = C_{\text{out}} \cdot C_{\text{in}} \cdot k_h \cdot k_w \cdot \text{sizeof}(\text{dtype})$$

### Activation Memory

During inference, intermediate activations consume memory proportional to the spatial dimensions:

$$M_{\text{act}}^{(l)} = N \cdot C_l \cdot H_l \cdot W_l \cdot \text{sizeof}(\text{dtype})$$

The **peak activation memory** depends on the execution order. For a sequential network:

$$M_{\text{peak\_act}} = \max_l \left( M_{\text{act}}^{(l)} + M_{\text{act}}^{(l+1)} \right)$$

This is because at layer $l$, we need both the input (from layer $l-1$) and the output (for layer $l+1$) simultaneously.

### Worked Example: MobileNetV2 on Raspberry Pi 4

| Component | Calculation | Memory |
|-----------|-------------|--------|
| Parameters (FP32) | 3.4M × 4 bytes | 13.6 MB |
| Parameters (INT8) | 3.4M × 1 byte | 3.4 MB |
| Peak activations (224×224, FP32) | 32 × 112 × 112 × 4 | 6.3 MB |
| ORT runtime overhead | — | ~50 MB |
| **Total (FP32)** | — | **~70 MB** |
| **Total (INT8)** | — | **~60 MB** |

The Raspberry Pi 4 has 1–8 GB RAM. With OS overhead (~400 MB), you have ample room for MobileNetV2 but not for ResNet-152 (60M params = 240 MB FP32).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Memory budget analysis for common edge models
models = {
    'MobileNetV2': {'params_M': 3.4, 'peak_act_MB': 6.3},
    'EfficientNet-B0': {'params_M': 5.3, 'peak_act_MB': 12.1},
    'ResNet-18': {'params_M': 11.7, 'peak_act_MB': 8.4},
    'ResNet-50': {'params_M': 25.6, 'peak_act_MB': 14.2},
    'YOLOv5s': {'params_M': 7.2, 'peak_act_MB': 18.5},
    'ResNet-152': {'params_M': 60.2, 'peak_act_MB': 22.1},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Memory breakdown by precision
names = list(models.keys())
params_fp32 = [m['params_M'] * 4 for m in models.values()]  # MB
params_int8 = [m['params_M'] * 1 for m in models.values()]
activations = [m['peak_act_MB'] for m in models.values()]
runtime_overhead = [50] * len(models)  # ORT baseline

x = np.arange(len(names))
width = 0.35

bars1 = axes[0].bar(x - width/2, params_fp32, width, label='Params (FP32)', color='#e74c3c')
bars2 = axes[0].bar(x + width/2, params_int8, width, label='Params (INT8)', color='#3498db')
axes[0].bar(x - width/2, activations, width, bottom=params_fp32, label='Activations', color='#f39c12', alpha=0.7)
axes[0].bar(x + width/2, activations, width, bottom=params_int8, color='#f39c12', alpha=0.7)

# Device memory limits
axes[0].axhline(y=1000, color='green', linestyle='--', alpha=0.7, label='RPi4 1GB')
axes[0].axhline(y=4000, color='purple', linestyle='--', alpha=0.7, label='Jetson Nano 4GB')

axes[0].set_xlabel('Model')
axes[0].set_ylabel('Memory (MB)')
axes[0].set_title('Model Memory Footprint vs Device Limits')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_ylim(0, 350)

# Right: Memory savings from quantization
savings = [(fp - i8) / fp * 100 for fp, i8 in zip(params_fp32, params_int8)]
colors = ['#2ecc71' if s > 70 else '#f1c40f' for s in savings]
axes[1].barh(names, savings, color=colors)
axes[1].set_xlabel('Parameter Memory Reduction (%)')
axes[1].set_title('Memory Savings: FP32 → INT8 Quantization')
axes[1].set_xlim(0, 100)
for i, v in enumerate(savings):
    axes[1].text(v + 1, i, f'{v:.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('edge_memory_budget.png', dpi=150, bbox_inches='tight')
plt.show()
print("Memory budget analysis complete.")

<a id='3'></a>
## 3. Power and Thermal Modeling

Edge devices operate under strict power budgets — often battery-powered or passively cooled. Understanding the physics of power consumption is essential for sustained inference.

### CMOS Dynamic Power

The dominant power consumption in digital circuits follows:

$$P_{\text{dynamic}} = \alpha \cdot C_{\text{eff}} \cdot V_{\text{dd}}^2 \cdot f$$

where:
- $\alpha$ = switching activity factor (fraction of gates toggling per cycle)
- $C_{\text{eff}}$ = effective switched capacitance
- $V_{\text{dd}}$ = supply voltage
- $f$ = clock frequency

The **quadratic voltage dependence** is why voltage scaling (DVFS) is so effective. Reducing voltage by 20% cuts dynamic power by 36%.

### Total System Power

$$P_{\text{total}} = P_{\text{dynamic}} + P_{\text{static}} + P_{\text{memory}} + P_{\text{IO}}$$

$$P_{\text{static}} = I_{\text{leak}} \cdot V_{\text{dd}} \propto e^{-V_{\text{th}} / (n \cdot V_T)}$$

Leakage power grows exponentially with temperature, creating a positive feedback loop: more compute → more heat → more leakage → more heat.

### Thermal Throttling Model

The junction temperature evolves as:

$$T_j(t) = T_{\text{amb}} + P_{\text{total}} \cdot R_{\theta_{JA}} \cdot \left(1 - e^{-t/\tau}\right)$$

where $R_{\theta_{JA}}$ is junction-to-ambient thermal resistance and $\tau = R_{\theta} \cdot C_{\theta}$ is the thermal time constant.

When $T_j$ exceeds the throttle threshold $T_{\text{throttle}}$, the processor reduces frequency:

$$f_{\text{throttled}} = f_{\text{max}} \cdot \frac{T_{\text{max}} - T_j}{T_{\text{max}} - T_{\text{throttle}}}$$

This means **sustained inference throughput** is lower than burst throughput — always benchmark after thermal steady-state (typically 5–10 minutes).

### Battery Life Estimation

For a battery-powered edge device:

$$t_{\text{battery}} = \frac{E_{\text{battery}}}{P_{\text{avg}}} = \frac{C \cdot V_{\text{nom}}}{P_{\text{inference}} \cdot \text{duty\_cycle} + P_{\text{idle}} \cdot (1 - \text{duty\_cycle})}$$

**Worked Example:** A 10,000 mAh (3.7V) battery with a Jetson Nano (10W inference, 2W idle) at 50% duty cycle:

$$t = \frac{10 \cdot 3.7}{10 \cdot 0.5 + 2 \cdot 0.5} = \frac{37}{6} \approx 6.2 \text{ hours}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thermal throttling simulation
T_amb = 25.0  # Ambient temperature (°C)
P_total = 10.0  # Power dissipation (W)
R_theta = 5.0  # Thermal resistance (°C/W) - fanless
C_theta = 20.0  # Thermal capacitance (J/°C)
tau = R_theta * C_theta  # Thermal time constant (s)
T_throttle = 70.0  # Throttle threshold
T_max = 85.0  # Maximum junction temperature
f_max = 1.5  # Max frequency (GHz)

t = np.linspace(0, 600, 1000)  # 10 minutes
T_j = T_amb + P_total * R_theta * (1 - np.exp(-t / tau))

# Throttled frequency
f_actual = np.where(
    T_j < T_throttle,
    f_max,
    f_max * np.clip((T_max - T_j) / (T_max - T_throttle), 0.3, 1.0)
)

# Relative throughput (proportional to frequency)
throughput_relative = f_actual / f_max * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Temperature plot
axes[0].plot(t, T_j, 'r-', linewidth=2, label='Junction Temperature')
axes[0].axhline(y=T_throttle, color='orange', linestyle='--', label=f'Throttle Threshold ({T_throttle}°C)')
axes[0].axhline(y=T_max, color='red', linestyle='--', alpha=0.5, label=f'Max Temperature ({T_max}°C)')
axes[0].axhline(y=T_amb, color='blue', linestyle=':', alpha=0.5, label=f'Ambient ({T_amb}°C)')
axes[0].fill_between(t, T_throttle, T_j, where=T_j > T_throttle, alpha=0.2, color='red', label='Throttled Region')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Edge Device Thermal Throttling Under Sustained Load')
axes[0].legend(loc='lower right')
axes[0].set_ylim(20, 90)
axes[0].grid(True, alpha=0.3)

# Throughput plot
axes[1].plot(t, throughput_relative, 'b-', linewidth=2, label='Relative Throughput')
axes[1].fill_between(t, throughput_relative, 100, alpha=0.15, color='red', label='Performance Loss')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Throughput (% of peak)')
axes[1].set_title('Inference Throughput Degradation from Thermal Throttling')
axes[1].legend(loc='lower left')
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3)

# Annotations
throttle_time = tau * np.log(P_total * R_theta / (P_total * R_theta - (T_throttle - T_amb)))
axes[0].axvline(x=throttle_time, color='gray', linestyle=':', alpha=0.7)
axes[0].annotate(f'Throttling begins\nt = {throttle_time:.0f}s',
                 xy=(throttle_time, T_throttle), xytext=(throttle_time + 40, T_throttle - 8),
                 arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.tight_layout()
plt.savefig('thermal_throttling.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Thermal time constant τ = {tau:.0f}s")
print(f"Steady-state junction temp = {T_amb + P_total * R_theta:.1f}°C")
print(f"Throttling begins at t = {throttle_time:.0f}s")

<a id='4'></a>
## 4. Compute Roofline Analysis

The **roofline model** characterizes the performance bound of a computation on a given processor. It reveals whether an operation is **compute-bound** or **memory-bound** — a critical distinction for edge optimization.

### Arithmetic Intensity

$$I = \frac{\text{FLOPs}}{\text{Bytes transferred}} \quad \left[\frac{\text{FLOP}}{\text{Byte}}\right]$$

### Performance Bound

$$\text{Attainable Performance} = \min\left(\pi, \beta \cdot I\right)$$

where $\pi$ is peak compute (FLOPS) and $\beta$ is memory bandwidth (Bytes/s).

The **ridge point** separates memory-bound from compute-bound regimes:

$$I_{\text{ridge}} = \frac{\pi}{\beta}$$

### Edge Device Examples

| Device | Peak Compute $\pi$ | Bandwidth $\beta$ | Ridge Point $I_{\text{ridge}}$ |
|--------|-------------------|-------------------|-------------------------------|
| RPi 4 (Cortex-A72) | 13.8 GFLOPS | 4.3 GB/s | 3.2 FLOP/Byte |
| Jetson Nano | 472 GFLOPS (FP16) | 25.6 GB/s | 18.4 FLOP/Byte |
| Jetson Orin Nano | 1.3 TFLOPS (INT8) | 68 GB/s | 19.1 FLOP/Byte |

### Implications for Layer Types

- **Depthwise convolutions** ($I \approx 2k^2$): Usually memory-bound on edge devices. For $3 \times 3$: $I \approx 18$ FLOP/Byte.
- **Pointwise (1×1) convolutions**: Higher arithmetic intensity, often compute-bound.
- **Fully connected layers** with small batch: $I \approx 2$ (severely memory-bound at batch=1).

This explains why MobileNet architectures (heavy on depthwise separable convolutions) are **memory-bandwidth limited** on edge devices — quantization helps by reducing bytes transferred, not just compute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Roofline model for edge devices
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

devices = {
    'Raspberry Pi 4\n(Cortex-A72)': {'pi': 13.8, 'beta': 4.3, 'color': '#e74c3c'},
    'Jetson Nano\n(Maxwell 128-core)': {'pi': 472, 'beta': 25.6, 'color': '#2ecc71'},
    'Jetson Orin Nano\n(Ampere)': {'pi': 1300, 'beta': 68, 'color': '#3498db'},
}

I_range = np.logspace(-1, 3, 500)  # Arithmetic intensity range

for name, d in devices.items():
    perf = np.minimum(d['pi'], d['beta'] * I_range)
    ax.loglog(I_range, perf, linewidth=2.5, color=d['color'], label=name)
    ridge = d['pi'] / d['beta']
    ax.axvline(x=ridge, color=d['color'], linestyle=':', alpha=0.4)

# Plot common operations
ops = {
    'FC (batch=1)': (2, 'o'),
    'Depthwise 3×3': (18, 's'),
    'Pointwise 1×1\n(C=256)': (128, '^'),
    'Conv 3×3\n(C=64→64)': (72, 'D'),
}

for op_name, (intensity, marker) in ops.items():
    ax.axvline(x=intensity, color='gray', linestyle='--', alpha=0.3)
    ax.text(intensity, 2000, op_name, ha='center', va='bottom', fontsize=8,
            rotation=0, color='gray')

ax.set_xlabel('Arithmetic Intensity (FLOP/Byte)', fontsize=12)
ax.set_ylabel('Attainable Performance (GFLOPS)', fontsize=12)
ax.set_title('Roofline Model for Edge Devices', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0.1, 1000)
ax.set_ylim(0.1, 5000)
ax.grid(True, alpha=0.3, which='both')
ax.fill_between([0.1, 3.2], [0.1, 0.1], [5000, 5000], alpha=0.05, color='red', label='Memory-bound zone (RPi4)')

plt.tight_layout()
plt.savefig('roofline_edge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Operations left of the ridge point are memory-bound.")
print("Quantization helps memory-bound ops by reducing data movement.")

<a id='5'></a>
## 5. Quantization for Edge Deployment

Quantization is the single most impactful optimization for edge deployment. It maps floating-point tensors to lower-precision integer representations, simultaneously reducing memory footprint, bandwidth consumption, and compute cost (via integer arithmetic units).

### Uniform Affine Quantization

The mapping from floating-point $r$ to quantized integer $q$ is:

$$q = \text{clamp}\left(\left\lfloor \frac{r}{s} \right\rceil + z, \; q_{\min}, \; q_{\max}\right)$$

$$r = s \cdot (q - z)$$

where:
- $s$ = scale factor: $s = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}$
- $z$ = zero point: $z = q_{\min} - \left\lfloor \frac{r_{\min}}{s} \right\rceil$
- For INT8: $q_{\min} = -128$, $q_{\max} = 127$ (symmetric) or $q_{\min} = 0$, $q_{\max} = 255$ (asymmetric)

### Quantization Error Analysis

The rounding error for uniform quantization follows:

$$\text{MSE}_{\text{round}} = \frac{\Delta^2}{12}$$

where $\Delta = s$ is the step size. This is the quantization noise for a uniform distribution within one step.

The **clipping error** from saturating values outside $[r_{\min}, r_{\max}]$:

$$\text{MSE}_{\text{clip}} = 2\int_{\alpha}^{\infty} (w - \alpha)^2 p(w) \, dw$$

For Gaussian weights with $w \sim \mathcal{N}(0, \sigma^2)$ and clipping threshold $\alpha$:

$$\text{MSE}_{\text{clip}} = 2\sigma^2 \left[ (1 + t^2)\Phi(-t) - t\phi(t) \right]$$

where $t = \alpha/\sigma$, $\Phi$ is the standard normal CDF, and $\phi$ is the PDF.

### Total Quantization Error

$$\text{MSE}_{\text{total}} = \underbrace{\frac{\Delta^2}{12}}_{\text{rounding}} + \underbrace{2\int_\alpha^\infty (w-\alpha)^2 p(w) \, dw}_{\text{clipping}}$$

The **optimal clipping threshold** minimizes total MSE. For Gaussian distributions, the optimal $\alpha \approx 2.83\sigma$ for 8-bit quantization — this is what calibration algorithms (entropy, percentile, MSE minimization) aim to find.

### Effective Bits with Group Quantization

When using per-group quantization with group size $g$, each group stores its own scale and zero-point (in FP16):

$$\text{Effective bits per weight} = b + \frac{16 \cdot 2}{g} = b + \frac{32}{g}$$

For INT4 with group size 128: $4 + 32/128 = 4.25$ bits per weight.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Quantization error analysis
sigma = 0.05  # Typical weight standard deviation
n_bits_range = np.arange(2, 9)

def compute_quantization_errors(sigma, n_bits, n_sigma_clip=3.0):
    """Compute rounding and clipping MSE for Gaussian weights."""
    alpha = n_sigma_clip * sigma
    n_levels = 2**n_bits
    delta = 2 * alpha / (n_levels - 1)  # Step size
    
    # Rounding error
    mse_round = delta**2 / 12
    
    # Clipping error for Gaussian
    t = alpha / sigma
    mse_clip = 2 * sigma**2 * ((1 + t**2) * stats.norm.cdf(-t) - t * stats.norm.pdf(t))
    
    return mse_round, mse_clip, delta

# Sweep over clipping thresholds for 8-bit
clip_range = np.linspace(1.5, 5.0, 200) * sigma
mse_round_sweep = []
mse_clip_sweep = []

for alpha in clip_range:
    delta = 2 * alpha / 255
    mse_r = delta**2 / 12
    t = alpha / sigma
    mse_c = 2 * sigma**2 * ((1 + t**2) * stats.norm.cdf(-t) - t * stats.norm.pdf(t))
    mse_round_sweep.append(mse_r)
    mse_clip_sweep.append(mse_c)

mse_round_sweep = np.array(mse_round_sweep)
mse_clip_sweep = np.array(mse_clip_sweep)
mse_total_sweep = mse_round_sweep + mse_clip_sweep

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Clipping threshold optimization
axes[0].semilogy(clip_range / sigma, mse_round_sweep, 'b-', label='Rounding MSE', linewidth=2)
axes[0].semilogy(clip_range / sigma, mse_clip_sweep, 'r-', label='Clipping MSE', linewidth=2)
axes[0].semilogy(clip_range / sigma, mse_total_sweep, 'k-', label='Total MSE', linewidth=2.5)

opt_idx = np.argmin(mse_total_sweep)
opt_alpha = clip_range[opt_idx] / sigma
axes[0].axvline(x=opt_alpha, color='green', linestyle='--', alpha=0.7, label=f'Optimal α = {opt_alpha:.2f}σ')
axes[0].scatter([opt_alpha], [mse_total_sweep[opt_idx]], color='green', s=100, zorder=5)

axes[0].set_xlabel('Clipping Threshold (multiples of σ)')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('INT8 Quantization Error vs Clipping Threshold\n(Gaussian weights, σ=0.05)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Error vs bit-width
total_errors = []
for n_bits in n_bits_range:
    mse_r, mse_c, _ = compute_quantization_errors(sigma, n_bits, 2.83)
    total_errors.append(mse_r + mse_c)

axes[1].semilogy(n_bits_range, total_errors, 'ko-', linewidth=2, markersize=8)
axes[1].set_xlabel('Quantization Bit-width')
axes[1].set_ylabel('Total MSE')
axes[1].set_title('Quantization Error vs Precision')
axes[1].grid(True, alpha=0.3)

for i, (bits, err) in enumerate(zip(n_bits_range, total_errors)):
    axes[1].annotate(f'{bits}-bit', (bits, err), textcoords='offset points',
                     xytext=(10, 5), fontsize=9)

plt.tight_layout()
plt.savefig('quantization_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Optimal clipping threshold: {opt_alpha:.2f}σ")
print(f"Minimum total MSE at optimal clip: {mse_total_sweep[opt_idx]:.2e}")

<a id='6'></a>
## 6. Hardware Platform Deep Dive

### Raspberry Pi 4/5 (ARM Cortex-A72/A76)

The Raspberry Pi represents the most accessible edge platform. Its ARM Cortex cores support NEON SIMD (128-bit), enabling 4 FP32 or 16 INT8 operations per cycle per lane.

**Theoretical INT8 throughput:**

$$\text{TOPS}_{\text{INT8}} = N_{\text{cores}} \times \frac{\text{SIMD\_width}}{\text{sizeof(INT8)}} \times 2 \times f_{\text{clock}}$$

For RPi4 (4 cores, 128-bit NEON, 1.5 GHz):

$$\text{TOPS} = 4 \times 16 \times 2 \times 1.5 \times 10^9 = 192 \text{ GOPS}$$

(In practice, ~30-50% utilization due to memory bandwidth limits.)

### NVIDIA Jetson Family

```
┌─────────────────────────────────────────────────────────────┐
│                    Jetson Module                              │
├─────────────────┬───────────────────────────────────────────┤
│   ARM CPU       │           GPU (CUDA Cores)                │
│   Cortex-A78    │   ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐      │
│   6-12 cores    │   │ SM0 │ │ SM1 │ │ SM2 │ │ ... │      │
│                 │   └──┬──┘ └──┬──┘ └──┬──┘ └──┬──┘      │
│                 │      └───────┴───────┴───────┘           │
│                 │              Shared L2                     │
├─────────────────┴───────────────────────────────────────────┤
│              Unified Memory (LPDDR5)                          │
│              Shared between CPU and GPU                       │
└─────────────────────────────────────────────────────────────┘
```

The **unified memory architecture** means CPU and GPU share the same physical memory — no PCIe copy overhead, but bandwidth contention exists.

| Jetson Model | GPU Cores | Peak INT8 | Memory | TDP |
|-------------|-----------|-----------|--------|-----|
| Nano (Maxwell) | 128 | 0.5 TOPS | 4 GB | 5-10W |
| Xavier NX | 384 (Volta) | 21 TOPS | 8 GB | 10-20W |
| Orin Nano | 1024 (Ampere) | 40 TOPS | 8 GB | 7-15W |
| Orin NX | 1024 (Ampere) | 100 TOPS | 8-16 GB | 10-25W |

### Intel Neural Compute Stick 2 (Myriad X VPU)

The Myriad X is a dedicated vision processing unit with 16 SHAVE cores optimized for fixed-point neural network inference. It achieves ~1 TOPS in a 1.5W envelope — excellent performance-per-watt for vision tasks.

**Key limitation:** The VPU has only 512 KB on-chip SRAM. Large activations must be tiled, adding scheduling complexity. Models with large spatial activations (e.g., high-resolution segmentation) may need architecture changes.

<a id='7'></a>
## 7. ONNX Runtime Execution Providers for Edge

ORT's **Execution Provider (EP)** abstraction decouples the model from the hardware backend. The same ONNX model can target different accelerators by changing the EP selection.

### Provider Selection Strategy

```
┌─────────────────────────────────────────────────────────────┐
│                    ORT Session                                │
├─────────────────────────────────────────────────────────────┤
│  Graph Partitioning: assign subgraphs to providers          │
├──────┬──────────┬──────────────┬───────────────────────────┤
│ Node │ TensorRT │   CUDA EP    │      CPU EP (fallback)    │
│  1   │    ✓     │              │                           │
│  2   │    ✓     │              │                           │
│  3   │          │      ✓       │                           │
│  4   │          │              │           ✓               │
│  5   │    ✓     │              │                           │
└──────┴──────────┴──────────────┴───────────────────────────┘
```

When multiple EPs are specified, ORT uses a **priority fallback chain**: each node is assigned to the highest-priority EP that supports it. Unsupported nodes fall through to CPU EP.

### Provider Performance Characteristics

| EP | Best For | Latency Profile | Memory Overhead |
|----|----------|----------------|------------------|
| CPU | ARM/x86 general | Predictable | Low |
| CUDA | Jetson GPU | High throughput, startup cost | Medium |
| TensorRT | Jetson optimized | Best throughput after warmup | High (engine cache) |
| OpenVINO | Intel VPU/iGPU | Good for vision | Medium |
| NNAPI | Android NPU | Varies by SoC | Low |
| CoreML | Apple Neural Engine | Excellent on Apple Silicon | Low |

### Thread Configuration for Edge

On multi-core edge devices, thread configuration significantly impacts both latency and power:

$$T_{\text{inference}} \approx \frac{\text{FLOPs}}{N_{\text{threads}} \times \text{FLOPS\_per\_thread}} + T_{\text{sync\_overhead}} \times \log_2(N_{\text{threads}})$$

The optimal thread count balances parallelism against synchronization overhead and thermal pressure. For a 4-core ARM device running continuous inference, 2-3 intra-op threads often outperforms 4 due to thermal headroom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate thread scaling on edge device
np.random.seed(42)

threads = np.arange(1, 9)
base_flops = 1e9  # 1 GFLOP workload
flops_per_thread = 3e9  # 3 GFLOPS per thread (ARM Cortex-A72 @ 1.5GHz)

# Ideal scaling
t_ideal = base_flops / (threads * flops_per_thread) * 1000  # ms

# Real scaling with sync overhead and thermal throttling
sync_overhead = 0.1 * np.log2(np.maximum(threads, 1))  # ms
thermal_penalty = np.where(threads > 3, 1.0 + 0.15 * (threads - 3), 1.0)
t_real = (base_flops / (threads * flops_per_thread) * 1000 + sync_overhead) * thermal_penalty

# Power consumption model
p_idle = 2.0  # Watts
p_per_thread = 1.2  # Watts per active core
power = p_idle + threads * p_per_thread * np.where(threads > 4, 1.1, 1.0)  # throttle adds leakage

# Energy per inference
energy = power * t_real / 1000  # Joules

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Latency vs threads
axes[0].plot(threads, t_ideal, 'b--', linewidth=1.5, label='Ideal scaling', alpha=0.6)
axes[0].plot(threads, t_real, 'r-o', linewidth=2, markersize=6, label='Real (thermal + sync)')
axes[0].axvline(x=4, color='gray', linestyle=':', alpha=0.5, label='Physical cores (RPi4)')
opt_threads = threads[np.argmin(t_real)]
axes[0].scatter([opt_threads], [t_real[opt_threads-1]], color='green', s=150, zorder=5, label=f'Optimal: {opt_threads} threads')
axes[0].set_xlabel('Number of Threads')
axes[0].set_ylabel('Inference Latency (ms)')
axes[0].set_title('Thread Scaling on Edge Device')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Power vs threads
axes[1].bar(threads, power, color='#e74c3c', alpha=0.7)
axes[1].set_xlabel('Number of Threads')
axes[1].set_ylabel('Power (Watts)')
axes[1].set_title('Power Consumption vs Thread Count')
axes[1].grid(True, alpha=0.3)

# Energy efficiency
axes[2].plot(threads, energy * 1000, 'g-s', linewidth=2, markersize=6)
axes[2].set_xlabel('Number of Threads')
axes[2].set_ylabel('Energy per Inference (mJ)')
axes[2].set_title('Energy Efficiency (Lower is Better)')
axes[2].grid(True, alpha=0.3)
opt_energy_threads = threads[np.argmin(energy)]
axes[2].scatter([opt_energy_threads], [energy[opt_energy_threads-1]*1000], color='green', s=150, zorder=5,
               label=f'Most efficient: {opt_energy_threads} threads')
axes[2].legend()

plt.tight_layout()
plt.savefig('thread_scaling_edge.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Optimal for latency: {opt_threads} threads ({t_real[opt_threads-1]:.2f} ms)")
print(f"Optimal for energy: {opt_energy_threads} threads ({energy[opt_energy_threads-1]*1000:.2f} mJ)")

<a id='8'></a>
## 8. Latency Decomposition and Pipelining

Understanding where time is spent in the inference pipeline is critical for optimization. On edge devices, preprocessing and postprocessing can dominate total latency for small models.

### End-to-End Latency Model

$$T_{\text{e2e}} = T_{\text{capture}} + T_{\text{decode}} + T_{\text{resize}} + T_{\text{normalize}} + T_{\text{infer}} + T_{\text{postprocess}} + T_{\text{render}}$$

### Pipeline Parallelism

Without pipelining, throughput is limited by total latency:

$$\text{Throughput}_{\text{serial}} = \frac{1}{T_{\text{e2e}}}$$

With N-stage pipelining:

$$\text{Throughput}_{\text{pipelined}} = \frac{1}{\max_i(T_{\text{stage}_i})}$$

The speedup is bounded by the slowest stage (the **bottleneck**):

$$\text{Speedup} = \frac{T_{\text{e2e}}}{\max_i(T_{\text{stage}_i})} \leq N$$

### Pipelining Architecture

```
Time →  t0    t1    t2    t3    t4    t5    ...
       ┌─────┐
Frame0 │Capt │Prep │Infer│Post │Disp │
       └─────┘─────┘─────┘─────┘─────┘
              ┌─────┐
Frame1        │Capt │Prep │Infer│Post │Disp │
              └─────┘─────┘─────┘─────┘─────┘
                     ┌─────┐
Frame2               │Capt │Prep │Infer│Post │Disp │
                     └─────┘─────┘─────┘─────┘─────┘

Throughput = 1/max(stage_time), not 1/sum(stage_times)
```

On a 4-core ARM device, you can dedicate:
- Core 0: Camera capture + decode
- Cores 1-2: ORT inference (intra_op_num_threads=2)
- Core 3: Postprocessing + display/actuation

### IOBinding for Zero-Copy Inference

On Jetson (unified memory), ORT's IOBinding API eliminates CPU↔GPU memory copies:

$$T_{\text{copy}} = \frac{\text{tensor\_size}}{\text{bandwidth}} \times 2 \;\text{(input + output)}$$

For a 224×224×3 FP32 input on Jetson Nano (25.6 GB/s shared):

$$T_{\text{copy}} = \frac{224 \times 224 \times 3 \times 4}{25.6 \times 10^9} \times 2 \approx 0.047 \text{ ms}$$

Small per tensor, but with multi-output models and high FPS, these copies add up. IOBinding pre-allocates GPU-resident buffers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Latency decomposition for edge inference pipeline
stages = ['Capture', 'Decode', 'Resize', 'Normalize', 'Inference', 'NMS/Post', 'Render']

# Simulated latencies (ms) for different models on RPi4
mobilenet_latency = [1.0, 2.5, 1.2, 0.8, 28.0, 0.5, 1.0]
yolov5n_latency = [1.0, 2.5, 1.5, 0.8, 85.0, 12.0, 2.0]
efficientnet_latency = [1.0, 2.5, 1.2, 0.8, 42.0, 0.5, 1.0]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Top: Waterfall chart for each model
x = np.arange(len(stages))
width = 0.25

bars1 = axes[0].bar(x - width, mobilenet_latency, width, label='MobileNetV2 (Classification)', color='#3498db')
bars2 = axes[0].bar(x, yolov5n_latency, width, label='YOLOv5n (Detection)', color='#e74c3c')
bars3 = axes[0].bar(x + width, efficientnet_latency, width, label='EfficientNet-B0 (Classification)', color='#2ecc71')

axes[0].set_xlabel('Pipeline Stage')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency Decomposition: Edge Inference Pipeline on Raspberry Pi 4')
axes[0].set_xticks(x)
axes[0].set_xticklabels(stages)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_yscale('log')
axes[0].set_ylim(0.3, 200)

# Bottom: Pipeline timing diagram
n_frames = 5
colors = plt.cm.Set3(np.linspace(0, 1, len(stages)))
stage_times = mobilenet_latency

# Serial execution
serial_starts = []
for frame in range(n_frames):
    frame_start = frame * sum(stage_times)
    cumulative = frame_start
    for s_time in stage_times:
        serial_starts.append(cumulative)
        cumulative += s_time

# Pipelined execution
bottleneck = max(stage_times)
pipe_starts = []
for frame in range(n_frames):
    frame_offset = frame * bottleneck
    cumulative = frame_offset
    for s_time in stage_times:
        pipe_starts.append(cumulative)
        cumulative += s_time

# Plot Gantt chart for pipelined execution
for frame in range(n_frames):
    for s_idx, (s_name, s_time) in enumerate(zip(stages, stage_times)):
        idx = frame * len(stages) + s_idx
        axes[1].barh(frame, s_time, left=pipe_starts[idx], height=0.6,
                     color=colors[s_idx], edgecolor='white', linewidth=0.5)

axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Frame')
axes[1].set_title('Pipelined Execution (MobileNetV2 on RPi4)')
axes[1].set_yticks(range(n_frames))
axes[1].set_yticklabels([f'Frame {i}' for i in range(n_frames)])
axes[1].grid(True, alpha=0.3, axis='x')

# Legend for stages
legend_patches = [plt.Rectangle((0,0),1,1, color=colors[i]) for i in range(len(stages))]
axes[1].legend(legend_patches, stages, loc='upper right', ncol=4, fontsize=8)

serial_throughput = 1000 / sum(stage_times)
pipelined_throughput = 1000 / bottleneck
axes[1].text(0.02, 0.02, f'Serial: {serial_throughput:.1f} FPS | Pipelined: {pipelined_throughput:.1f} FPS | Speedup: {pipelined_throughput/serial_throughput:.1f}×',
            transform=axes[1].transAxes, fontsize=10, verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('latency_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Serial throughput: {serial_throughput:.1f} FPS")
print(f"Pipelined throughput: {pipelined_throughput:.1f} FPS")
print(f"Bottleneck stage: {stages[np.argmax(stage_times)]} ({max(stage_times):.1f} ms)")

<a id='9'></a>
## 9. Deployment Architecture Patterns

Edge deployments follow several architectural patterns depending on connectivity, latency requirements, and computational capacity.

### Pattern 1: Standalone Edge Inference

```
┌────────────────────────────────────────┐
│           Edge Device                   │
│                                         │
│  ┌──────┐   ┌────────┐   ┌─────────┐  │
│  │Camera│──▶│Preproc │──▶│ORT Infer│  │
│  └──────┘   └────────┘   └────┬────┘  │
│                                │        │
│                           ┌────▼────┐   │
│                           │Postproc │   │
│                           └────┬────┘   │
│                                │        │
│                           ┌────▼────┐   │
│                           │Actuator │   │
│                           └─────────┘   │
└────────────────────────────────────────┘
```

**Use case:** Safety-critical systems requiring guaranteed latency (autonomous vehicles, industrial quality inspection). No network dependency.

### Pattern 2: Edge-Cloud Hybrid (Cascade)

```
┌──────────────────────┐         ┌──────────────────────┐
│     Edge Device       │         │    Cloud Backend      │
│                       │         │                       │
│  ┌──────┐  ┌──────┐  │  Low    │  ┌──────────────┐   │
│  │Sensor│─▶│Light │──┼──conf──▶│  │Heavy Model   │   │
│  └──────┘  │Model │  │  only   │  │(ResNet-152)  │   │
│            └──┬───┘  │         │  └──────────────┘   │
│               │      │         │                       │
│          High conf   │         └──────────────────────┘
│               │      │
│          ┌────▼───┐  │
│          │Actuate  │  │
│          └────────┘  │
└──────────────────────┘
```

**Decision function:** Send to cloud when confidence is below threshold:

$$\text{route} = \begin{cases} \text{edge} & \text{if } \max(\text{softmax}(\mathbf{z})) \geq \tau \\ \text{cloud} & \text{otherwise} \end{cases}$$

Expected cloud offload ratio:

$$R_{\text{cloud}} = P(\max(\text{softmax}(\mathbf{z})) < \tau)$$

### Pattern 3: Federated Edge Fleet

Multiple edge devices run inference independently but aggregate insights:

```
┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐
│Edge1│  │Edge2│  │Edge3│  │EdgeN│
└──┬──┘  └──┬──┘  └──┬──┘  └──┬──┘
   │        │        │        │
   └────────┴────┬───┴────────┘
                 │
           ┌─────▼─────┐
           │Aggregation│
           │  Gateway   │
           └─────┬─────┘
                 │
           ┌─────▼─────┐
           │   Cloud    │
           │ (Analytics)│
           └───────────┘
```

### Model Update Strategy (OTA)

Updating models on edge fleets requires careful rollout:

1. **Canary deployment:** Update 1% of fleet, monitor accuracy/latency
2. **Gradual rollout:** Increase to 10%, 50%, 100% if metrics hold
3. **Rollback trigger:** Automatic revert if error rate exceeds baseline by $\delta$:

$$\text{rollback if } \frac{\text{errors}_{\text{new}}}{N_{\text{new}}} - \frac{\text{errors}_{\text{old}}}{N_{\text{old}}} > \delta$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Edge-Cloud cascade analysis
np.random.seed(42)

# Simulate model confidence scores
n_samples = 10000
# Mix of easy (high confidence) and hard (low confidence) samples
confidence_easy = np.random.beta(10, 2, size=int(n_samples * 0.7))
confidence_hard = np.random.beta(2, 5, size=int(n_samples * 0.3))
confidences = np.concatenate([confidence_easy, confidence_hard])

# Sweep threshold
thresholds = np.linspace(0.3, 0.99, 100)
cloud_ratios = [np.mean(confidences < tau) for tau in thresholds]
edge_latency = 30  # ms (light model)
cloud_latency = 150  # ms (network + heavy model)
avg_latencies = [edge_latency * (1 - cr) + cloud_latency * cr for cr in cloud_ratios]

# Accuracy model: cloud is more accurate on low-confidence samples
edge_accuracy = [0.85 + 0.1 * (1 - cr) for cr in cloud_ratios]  # worse on hard samples
hybrid_accuracy = [0.85 + 0.12 * (1 - cr) + 0.05 * cr for cr in cloud_ratios]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confidence distribution
axes[0].hist(confidences, bins=50, density=True, alpha=0.7, color='#3498db', edgecolor='white')
axes[0].axvline(x=0.7, color='red', linestyle='--', linewidth=2, label='τ = 0.7')
axes[0].axvline(x=0.9, color='orange', linestyle='--', linewidth=2, label='τ = 0.9')
axes[0].set_xlabel('Max Softmax Confidence')
axes[0].set_ylabel('Density')
axes[0].set_title('Edge Model Confidence Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cloud offload ratio vs threshold
axes[1].plot(thresholds, [cr * 100 for cr in cloud_ratios], 'b-', linewidth=2)
axes[1].set_xlabel('Confidence Threshold (τ)')
axes[1].set_ylabel('Cloud Offload Ratio (%)')
axes[1].set_title('Fraction of Requests Sent to Cloud')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=20, color='green', linestyle=':', alpha=0.7, label='20% budget')
axes[1].legend()

# Latency-accuracy tradeoff
sc = axes[2].scatter(avg_latencies, hybrid_accuracy, c=thresholds, cmap='RdYlGn_r', s=20)
axes[2].set_xlabel('Average Latency (ms)')
axes[2].set_ylabel('System Accuracy')
axes[2].set_title('Latency-Accuracy Pareto Front')
plt.colorbar(sc, ax=axes[2], label='Threshold τ')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('edge_cloud_cascade.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"At τ=0.7: {np.mean(confidences < 0.7)*100:.1f}% to cloud, avg latency={edge_latency*(1-np.mean(confidences<0.7))+cloud_latency*np.mean(confidences<0.7):.0f}ms")
print(f"At τ=0.9: {np.mean(confidences < 0.9)*100:.1f}% to cloud, avg latency={edge_latency*(1-np.mean(confidences<0.9))+cloud_latency*np.mean(confidences<0.9):.0f}ms")

<a id='10'></a>
## 10. Reliability and Monitoring on Edge

Edge devices operate in uncontrolled environments — temperature swings, power fluctuations, and physical tampering. Production edge ML requires robust reliability mechanisms.

### Watchdog Architecture

A hardware/software watchdog ensures the inference loop doesn't hang:

$$T_{\text{watchdog}} > T_{\text{worst\_case\_inference}} + T_{\text{margin}}$$

Typically $T_{\text{watchdog}} = 3 \times T_{\text{p99\_inference}}$.

### Data Drift Detection

Edge models encounter distribution shift over time (lighting changes, sensor degradation). Monitor using running statistics:

$$\text{KL}(P_{\text{current}} \| P_{\text{reference}}) = \sum_x P_{\text{current}}(x) \log \frac{P_{\text{current}}(x)}{P_{\text{reference}}(x)}$$

Alert when $\text{KL} > \epsilon$ for the confidence distribution.

### Resource Monitoring Metrics

Essential telemetry for edge ML in production:

| Metric | Formula | Alert Threshold |
|--------|---------|------------------|
| Memory utilization | $U_{\text{mem}} = \frac{M_{\text{used}}}{M_{\text{total}}}$ | > 85% |
| CPU utilization | $U_{\text{cpu}} = 1 - \frac{T_{\text{idle}}}{T_{\text{total}}}$ | > 90% sustained |
| Inference p99 latency | $T_{\text{p99}}$ from sliding window | > 2× baseline |
| Thermal headroom | $T_{\text{throttle}} - T_{\text{current}}$ | < 5°C |
| Prediction entropy | $H = -\sum_i p_i \log p_i$ | Rising trend |

### A/B Testing on Edge Fleets

When deploying a new model version, statistical significance requires:

$$N \geq \frac{(z_{\alpha/2} + z_\beta)^2 \cdot 2\sigma^2}{\delta^2}$$

where $\delta$ is the minimum detectable effect, $\sigma^2$ is the metric variance, $\alpha$ is the significance level, and $\beta$ is the acceptable Type II error rate. For edge fleets with heterogeneous hardware, stratified sampling by device type is essential.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Drift detection simulation
np.random.seed(42)

# Reference distribution (training time)
n_reference = 5000
reference_confidences = np.random.beta(8, 2, size=n_reference)

# Simulate gradual drift over time (e.g., sensor degradation)
n_days = 30
n_per_day = 200
kl_divergences = []
daily_means = []

for day in range(n_days):
    # Distribution shifts gradually
    drift_factor = 1.0 + day * 0.05  # Increasing drift
    alpha_shifted = max(8 - day * 0.2, 2)
    beta_shifted = 2 + day * 0.1
    day_confidences = np.random.beta(alpha_shifted, beta_shifted, size=n_per_day)
    daily_means.append(np.mean(day_confidences))
    
    # Compute KL divergence using histogram approximation
    bins = np.linspace(0, 1, 50)
    ref_hist, _ = np.histogram(reference_confidences, bins=bins, density=True)
    cur_hist, _ = np.histogram(day_confidences, bins=bins, density=True)
    
    # Add small epsilon to avoid log(0)
    ref_hist = ref_hist + 1e-10
    cur_hist = cur_hist + 1e-10
    ref_hist = ref_hist / ref_hist.sum()
    cur_hist = cur_hist / cur_hist.sum()
    
    kl = np.sum(cur_hist * np.log(cur_hist / ref_hist))
    kl_divergences.append(kl)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# KL divergence over time
days = np.arange(n_days)
alert_threshold = 0.5
axes[0, 0].plot(days, kl_divergences, 'b-o', markersize=4, linewidth=1.5)
axes[0, 0].axhline(y=alert_threshold, color='red', linestyle='--', label=f'Alert Threshold (ε={alert_threshold})')
alert_day = next((d for d, kl in enumerate(kl_divergences) if kl > alert_threshold), None)
if alert_day:
    axes[0, 0].axvline(x=alert_day, color='red', linestyle=':', alpha=0.5)
    axes[0, 0].annotate(f'Alert! Day {alert_day}', xy=(alert_day, kl_divergences[alert_day]),
                        xytext=(alert_day+2, kl_divergences[alert_day]+0.1),
                        arrowprops=dict(arrowstyle='->', color='red'))
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('KL Divergence')
axes[0, 0].set_title('Distribution Drift Detection')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Mean confidence over time
axes[0, 1].plot(days, daily_means, 'g-o', markersize=4, linewidth=1.5)
axes[0, 1].axhline(y=np.mean(reference_confidences), color='blue', linestyle='--', label='Reference mean')
axes[0, 1].set_xlabel('Day')
axes[0, 1].set_ylabel('Mean Confidence')
axes[0, 1].set_title('Model Confidence Degradation')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Distribution comparison: Day 0 vs Day 25
day0_conf = np.random.beta(8, 2, size=1000)
day25_conf = np.random.beta(3, 2.5, size=1000)
axes[1, 0].hist(day0_conf, bins=40, alpha=0.6, density=True, label='Day 0 (baseline)', color='#3498db')
axes[1, 0].hist(day25_conf, bins=40, alpha=0.6, density=True, label='Day 25 (drifted)', color='#e74c3c')
axes[1, 0].set_xlabel('Confidence Score')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Confidence Distribution Shift')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Inference latency monitoring
base_latency = 30  # ms
latency_p50 = base_latency + np.random.randn(n_days) * 2 + days * 0.3  # Gradual increase
latency_p99 = latency_p50 * 1.8 + np.random.exponential(5, n_days)
axes[1, 1].fill_between(days, latency_p50, latency_p99, alpha=0.3, color='#3498db', label='p50-p99 range')
axes[1, 1].plot(days, latency_p50, 'b-', linewidth=2, label='p50 latency')
axes[1, 1].plot(days, latency_p99, 'r-', linewidth=1.5, label='p99 latency')
axes[1, 1].axhline(y=base_latency * 2, color='red', linestyle='--', alpha=0.5, label='SLA (2× baseline)')
axes[1, 1].set_xlabel('Day')
axes[1, 1].set_ylabel('Latency (ms)')
axes[1, 1].set_title('Inference Latency Monitoring')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('edge_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Drift alert triggered on day {alert_day}")
print(f"Reference mean confidence: {np.mean(reference_confidences):.3f}")
print(f"Final day mean confidence: {daily_means[-1]:.3f}")

## 11. Edge Deployment Checklist

### Pre-Deployment Validation

| Step | Verification | Pass Criteria |
|------|-------------|----------------|
| 1 | Model fits in memory | $M_{\text{total}} < 0.7 \times M_{\text{device}}$ |
| 2 | Latency within budget | $T_{\text{p99}} < T_{\text{budget}}$ |
| 3 | Accuracy parity | $|\text{acc}_{\text{edge}} - \text{acc}_{\text{cloud}}| < \epsilon$ |
| 4 | Thermal stability | Sustained 10-min test, no throttling |
| 5 | Power within envelope | $P_{\text{sustained}} < P_{\text{budget}}$ |
| 6 | OTA update path tested | Rollback verified |
| 7 | Monitoring telemetry flowing | Dashboard populated |

### Common Failure Modes

| Failure | Root Cause | Mitigation |
|---------|-----------|-------------|
| OOM at load | Model too large for device | Quantize, external data, smaller arch |
| Latency spike after 5 min | Thermal throttling | Reduce duty cycle, add heatsink |
| Accuracy collapse in field | Input distribution shift | Drift monitoring + retrain trigger |
| Intermittent crashes | Memory fragmentation | Pre-allocate buffers, restart schedule |

---

## Summary

Edge deployment is fundamentally different from cloud serving. The mathematical framework presented here — memory budgets, power modeling, roofline analysis, quantization error theory — provides the tools to make principled decisions about model selection, optimization, and deployment architecture. Key takeaways:

1. **Memory is king:** $M_{\text{total}} = M_{\text{params}} + M_{\text{activations}} + M_{\text{runtime}}$ must fit within device constraints
2. **Power drives design:** $P \propto V^2 f C$ — quantization reduces both compute and memory power
3. **Thermal limits throughput:** Sustained performance ≠ burst performance; always benchmark after steady-state
4. **Pipeline for throughput:** Decompose latency, pipeline stages, optimize the bottleneck
5. **Monitor everything:** Drift detection, latency percentiles, and thermal headroom are non-negotiable in production

In [ ]:
# Final summary: Edge deployment decision matrix
import numpy as np
import matplotlib.pyplot as plt

# Radar chart for device comparison
categories = ['Compute\n(TOPS)', 'Memory\n(GB)', 'Power Eff.\n(TOPS/W)', 'Cost\n(inverse)', 'Ecosystem', 'Availability']
N = len(categories)

# Normalized scores (0-1)
devices_scores = {
    'Raspberry Pi 4': [0.1, 0.5, 0.3, 0.9, 0.9, 0.95],
    'Jetson Nano': [0.4, 0.5, 0.5, 0.6, 0.7, 0.7],
    'Jetson Orin Nano': [0.8, 0.7, 0.8, 0.3, 0.7, 0.6],
    'Intel NCS2': [0.15, 0.1, 0.9, 0.7, 0.5, 0.4],
}

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
for (name, scores), color in zip(devices_scores.items(), colors):
    values = scores + scores[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Edge Device Comparison Radar', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('edge_device_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Edge Deployment Deep Dive complete.")
print("Key insight: Choose device based on your dominant constraint (compute, power, or cost).")